# B2：画面真的听从按键吗

这一次不满足于平均训练 loss。我们固定同一帧，只换动作；随后连续生成，并用一个小去噪器作方法对照。

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'src' / 'hwm').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
import torch
from hwm.data import make_pixelworld_dataset
from hwm.video import TinyVQVAE, ActionTokenTransformer, TinyConditionalDenoiser, video_batch_from_episodes
torch.manual_seed(1)


## 1. 训练一个很小的离散视频模型

这里重复 B1 的短训练，使 Notebook 可以单独运行。PA1-B 会保存 tokenizer，不会每次重训。

In [ ]:
episodes = make_pixelworld_dataset(12, 8, seed=2)
current, actions, following = video_batch_from_episodes(episodes)
images = torch.cat((current, following))
tokenizer = TinyVQVAE(codebook_size=16, embedding_size=8)
opt = torch.optim.Adam(tokenizer.parameters(), lr=1e-3)
for _ in range(30):
    opt.zero_grad(); loss, _ = tokenizer.continuous_loss(images); loss.backward(); opt.step()
tokenizer.initialize_codebook(images)
with torch.no_grad():
    current_tokens = tokenizer.encode_tokens(current).flatten(1); next_tokens = tokenizer.encode_tokens(following).flatten(1)
dynamics = ActionTokenTransformer(codebook_size=16, model_size=32)
opt = torch.optim.Adam(dynamics.parameters(), lr=3e-3)
for _ in range(40):
    opt.zero_grad(); loss = dynamics.loss(current_tokens, actions, next_tokens); loss.backward(); opt.step()
print('最后 token loss:', round(float(loss.detach()), 4))


## 2. 反事实动作

把完全相同的起点复制五份，只替换按键。若输出完全相同，模型可能只在复制画面。

In [ ]:
same_start = current_tokens[:1].expand(5, -1)
all_actions = torch.arange(5)
with torch.no_grad(): counterfactual_logits = dynamics(same_start, all_actions)
differences = [(counterfactual_logits[0] - counterfactual_logits[i]).abs().mean().item() for i in range(1, 5)]
print('与 stay 相比的 logits 差异:', [round(x, 4) for x in differences])
print('注意：logits 改变仍不等于 argmax token 和解码画面一定改变。')
assert max(differences) > 0


## 3. 让模型吃自己的输出

一步预测时输入来自真实数据；连续生成时，第二步开始输入来自模型自己。小错误会进入下一次预测。

In [ ]:
rollout = [current_tokens[:1]]
right = torch.tensor([2])
with torch.no_grad():
    for _ in range(8): rollout.append(dynamics(rollout[-1], right).argmax(-1))
changes = [int(not torch.equal(rollout[i], rollout[i+1])) for i in range(8)]
print('每一步 token 是否改变:', changes)
print('不同帧数不等于运动正确；PA1-B 还要解码并测方块位移。')


## 4. 连续表示的去噪对照

Diffusion 不先挑离散编号，而是从带噪连续图像逐步还原。下面只训练一次噪声预测，目的是看清接口和额外延迟，不冒充 DIAMOND。

In [ ]:
denoiser = TinyConditionalDenoiser()
opt = torch.optim.Adam(denoiser.parameters(), lr=2e-3)
noise_level = torch.full((len(current),), 0.2)
noise = torch.randn_like(following)
noisy = following + noise_level[:, None, None, None] * noise
losses = []
for _ in range(20):
    opt.zero_grad(); predicted = denoiser(noisy, current, actions, noise_level); loss = torch.nn.functional.mse_loss(predicted, noise); loss.backward(); opt.step(); losses.append(float(loss.detach()))
print('denoise loss:', round(losses[0], 3), '→', round(losses[-1], 3))
assert losses[-1] < losses[0]


## 小结

AR token 模型每次直接给出下一组编号；Diffusion 要进行若干次去噪。PA1-B 必须在同一数据和算力预算下比较动作一致性、长时漂移与延迟，而不是只挑最好看的截图。